In [ ]:
! pip install telepot

In [ ]:
import json
import time
import requests
import telepot
from telepot.loop import MessageLoop
import paho.mqtt.client as mqtt
import os
import math

In [ ]:
class NotificationBot:
    def __init__(self, settings, chatID):
        self.settings = settings
        self.token = settings["telegramToken"]
        self.chatID = chatID

        # MQTT Setup
        self.mqtt_broker = os.getenv('MQTT_BROKER_HOST', 'mosquitto')
        self.mqtt_port = int(os.getenv('MQTT_BROKER_PORT', 1883))
        self.topic_observation = "building/observation"
        self.topic_action = "building/action"

        # Buffer to store the latest data received from MQTT
        self.last_observation = {}
        self.last_action = {}

        # Timestamp of the last sent alert
        self.last_alert_sent = 0

        # Flat2 mapping (your scenario)
        # observation payload usually has Tin_Flat2_Zona* keys
        self.zone_temp_keys = {
            "Zone 1": ["Tin_Flat2_Zona1", "Tair_z1"],
            "Zone 2": ["Tin_Flat2_Zona2", "Tair_z2"],
            "Zone 3": ["Tin_Flat2_Zona3", "Tair_z3"],
            "Zone 4": ["Tin_Flat2_Zona4", "Tair_z4"],
        }

        # action payload can use RL naming (A_Shade_Flat2_...) or legacy FMU naming
        self.shades = {
            "Zone 1": [
                ("Wall 2", ["A_Shade_Flat2_Z1_W2", "Zona1_wall2_shade_FMU"]),
                ("Wall 8", ["A_Shade_Flat2_Z1_W8", "Zona1_wall8_shade_FMU"]),
                ("Wall 9", ["A_Shade_Flat2_Z1_W9", "Zona1_wall9_shade_FMU"]),
            ],
            "Zone 2": [
                ("Wall 2", ["A_Shade_Flat2_Z2_W2", "Zona2_wall2_shade_FMU"]),
                ("Wall 3", ["A_Shade_Flat2_Z2_W3", "Zona2_wall3_shade_FMU"]),
            ],
            "Zone 4": [
                ("Wall 2", ["A_Shade_Flat2_Z4_W2", "Zona4_wall2_shade_FMU"]),
            ]
        }

        # Comfort thresholds
        self.t_min = 19
        self.t_max = 24

        # Initialize MQTT Client
        self.mqtt_client = mqtt.Client()
        self.mqtt_client.on_connect = self.on_mqtt_connect
        self.mqtt_client.on_message = self.on_mqtt_message
        self.mqtt_client.connect(self.mqtt_broker, self.mqtt_port)
        self.mqtt_client.loop_start()

        self.bot = telepot.Bot(self.token)
        MessageLoop(self.bot, {'chat': self.on_chat_message}).run_as_thread()

    def _first_existing(self, data, keys, default=None):
        for k in keys:
            if k in data and data[k] is not None:
                return data[k]
        return default

    def on_mqtt_connect(self, client, userdata, flags, rc):
        print(f"✅ Bot connected to MQTT broker (rc={rc})")
        client.subscribe(self.topic_observation, qos=1)
        client.subscribe(self.topic_action, qos=1)

    def on_mqtt_message(self, client, userdata, msg):
        try:
            payload = json.loads(msg.payload)

            if msg.topic == self.topic_observation:
                self.last_observation.update(payload)
                self.monitor_and_alert(chat_id=self.chatID)

            elif msg.topic == self.topic_action:
                self.last_action.update(payload)

            else:
                print(f"Unknown topic received: {msg.topic}")

        except Exception as e:
            print(f"Error parsing MQTT message: {e}")

    def on_chat_message(self, msg):
        content_type, chat_type, chat_id = telepot.glance(msg)
        text = msg.get('text', '').strip().lower()

        if text == '/temperature':
            self.send_temperature_status(chat_id)
        elif text == '/shades':
            self.send_shading_status(chat_id)
        elif text == '/status':
            self.send_overall_status(chat_id)
        elif text == '/help':
            msg = (
                "Commands:\n"
                "/temperature - Get temperature for all zones\n"
                "/shades - Get current blind positions\n"
                "/status - Outdoor conditions (Tout, DNI)\n"
            )
            self.bot.sendMessage(chat_id, msg)

    def send_temperature_status(self, chat_id):
        """Fetch latest temperatures for all zones."""
        if not self.last_observation:
            self.bot.sendMessage(chat_id, "No data received yet via MQTT.")
            return

        status_msg = "🌡 **Current Temperatures (Flat2):**\n"
        for zone_name, keys in self.zone_temp_keys.items():
            temp = self._first_existing(self.last_observation, keys)
            if temp is not None:
                truncated_temp = math.floor(float(temp) * 10) / 10
                status_msg += f"- {zone_name}: {truncated_temp}°C\n"
            else:
                status_msg += f"- {zone_name}: N/A\n"

        self.bot.sendMessage(chat_id, status_msg, parse_mode='Markdown')

    def send_overall_status(self, chat_id):
        """Send quick weather/solar status."""
        if not self.last_observation:
            self.bot.sendMessage(chat_id, "No data received yet via MQTT.")
            return

        tout = self._first_existing(self.last_observation, ["Tout", "T_ext"])
        dni = self._first_existing(self.last_observation, ["DNI"])

        msg = "📊 **Current Outdoor Status:**\n"
        if tout is not None:
            msg += f"- Tout: {math.floor(float(tout) * 10) / 10}°C\n"
        else:
            msg += "- Tout: N/A\n"

        if dni is not None:
            msg += f"- DNI: {math.floor(float(dni) * 10) / 10}\n"
        else:
            msg += "- DNI: N/A\n"

        self.bot.sendMessage(chat_id, msg, parse_mode='Markdown')

    def send_shading_status(self, chat_id):
        """Check if blinds are Open (0) or Closed (>1)."""
        if not self.last_action:
            self.bot.sendMessage(chat_id, "No data received yet via MQTT.")
            return

        status_msg = "🪟 **Shading Status (Flat2):**\n"
        for zone, shade_list in self.shades.items():
            status_msg += f"🏠 **{zone}**\n"
            for label, keys in shade_list:
                status_val = self._first_existing(self.last_action, keys)
                if status_val is None:
                    status_msg += f"  - {label}: N/A\n"
                    continue

                status_float = float(status_val)
                icon = "🌑" if status_float > 1.0 else "☀️"
                text = "Closed" if status_float > 1.0 else "Open"
                status_msg += f"  - {label}: {text} {icon} ({status_float:.1f})\n"
            status_msg += "\n"

        self.bot.sendMessage(chat_id, status_msg, parse_mode='Markdown')

    def monitor_and_alert(self, chat_id):
        """Check all zones for comfort violations (t_min-t_max°C)."""
        if not self.last_observation:
            return

        # Only alert if 1 min have passed since the last one
        current_time = time.time()
        if current_time - self.last_alert_sent < 60:
            return

        uncomfort_zones = []
        for zone_name, keys in self.zone_temp_keys.items():
            temp = self._first_existing(self.last_observation, keys)
            if temp is None:
                continue

            temp_float = float(temp)
            truncated_temp = math.floor(temp_float * 10) / 10

            if truncated_temp < self.t_min or truncated_temp > self.t_max:
                uncomfort_zones.append((zone_name, truncated_temp))

        if uncomfort_zones:
            alert_msg = f"⚠️ **Comfort Alert!** Zones out of comfort range ({self.t_min}-{self.t_max}°C):\n"
            for zone_label, zone_temp in uncomfort_zones:
                alert_msg += f"- {zone_label}: {zone_temp}°C\n"

            self.last_alert_sent = current_time
            self.bot.sendMessage(chat_id, alert_msg, parse_mode='Markdown')

In [ ]:
with open("settings.json") as f:
    conf = json.load(f)

TEST_CHAT_ID = 7201915868

bot = NotificationBot(conf, TEST_CHAT_ID)
print("Notification Bot started...")

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Stopping Bot...")